# PicoCal — Experiment story (E0 to E12), runnable


 The same logic also lives in `scripts/run_experiments.py` and `scripts/run_robustness.py`.

Score = `sigma_eff` of `(E_reco-E_true)/E_true` (lower is better).

## Setup

In [1]:
import sys
from pathlib import Path
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor

repo = Path.cwd().resolve()
if repo.name == "notebooks":
    repo = repo.parent
sys.path.insert(0, str(repo / "scripts"))
from run_experiments import (build, split, resolution, train_eval, variant_tokens,
                             Transformer, DeepSets, EPS)
import torch

FILES = 100
EPOCHS = 30
SEEDS = 5
BATCH = 64
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
{"device": DEVICE, "files": FILES, "epochs": EPOCHS, "seeds": SEEDS}

{'device': 'cuda', 'files': 100, 'epochs': 30, 'seeds': 5}

## Build the R3 dataset

In [2]:
files = sorted((repo / "data" / "full").glob("matched_*.root"))[:FILES]
D = build(files, 3, 100.0)
y = D["y"]; Et = D["Etrue"]
ridx = np.flatnonzero(D["region"] == 3)
rtr, rva, rte = (ridx[s] for s in split(len(ridx)))
in_dim = D["tok_seed"][int(ridx[0])].shape[1]
{"R3": len(ridx), "train": len(rtr), "test": len(rte), "token_dim": in_dim}

{'R3': 10692, 'train': 7484, 'test': 1604, 'token_dim': 12}

## Feature layout


| model | features |
|---|---|
| **Transformer / Deep Sets** | **12 per cell**: 7 continuous + 5 one-hot region |
| **BDT (E0b)** | **6 per cluster** (aggregate): logSumE, front/back ratio, n cells, log seedE, lateral RMS, region |
| **B0 / B1 / B2** | one scalar (sum / seed / total_energy) |
| **E12 ablations** | the 12 with one group removed/changed (10 / 7 / 12) |

The 12 per-cell transformer features:

1. log(energy)  2. log(front)  3. log(back)  4. rel_x/pitch  5. rel_y/pitch  6. rel_dr/pitch  7. log(pitch)  8-12. one-hot region R0..R4

In [1]:
import sys
import pandas as pd
from pathlib import Path

repo = Path.cwd().resolve()
if repo.name == "notebooks":
    repo = repo.parent
sys.path.insert(0, str(repo / "scripts"))
from run_experiments import build

FEATURES = ["log(energy)", "log(front)", "log(back)", "rel_x/pitch", "rel_y/pitch",
            "rel_dr/pitch", "log(pitch)", "region_R0", "region_R1", "region_R2",
            "region_R3", "region_R4"]
demo = build(sorted((repo / "data" / "full").glob("matched_*.root"))[:1], 3, 100.0)
print("token shape (cells, features):", demo["tok_seed"][0].shape)
pd.DataFrame(demo["tok_seed"][0], columns=FEATURES).round(3)

token shape (cells, features): (9, 12)


,log(energy),log(front),log(back),rel_x/pitch,rel_y/pitch,rel_dr/pitch,log(pitch),region_R0,region_R1,region_R2,region_R3,region_R4
0,9.930,8.377,9.692,0.000,0.000,0.000,4.787,0.0,0.0,0.0,0.0,1.0
1,4.587,1.996,4.520,-1.016,-1.016,1.437,4.787,0.0,0.0,0.0,0.0,1.0
2,6.638,3.772,6.581,0.000,-1.016,1.016,4.787,0.0,0.0,0.0,0.0,1.0
3,4.592,0.187,4.590,1.016,-1.016,1.437,4.787,0.0,0.0,0.0,0.0,1.0
4,5.230,2.758,5.148,-1.016,0.000,1.016,4.787,0.0,0.0,0.0,0.0,1.0
5,5.381,3.786,5.160,1.016,0.000,1.016,4.787,0.0,0.0,0.0,0.0,1.0
6,3.532,0.000,3.532,-1.016,1.016,1.437,4.787,0.0,0.0,0.0,0.0,1.0
7,4.791,3.342,4.534,0.000,1.016,1.016,4.787,0.0,0.0,0.0,0.0,1.0
8,3.338,2.880,2.430,1.016,1.016,1.437,4.787,0.0,0.0,0.0,0.0,1.0


## E0 — analytic baselines (calibrated to GeV)

Each raw energy proxy is affine-fit on train, then scored on test.

In [3]:
def calib(xtr, xte, ytr, Ete):
    a, b = np.polyfit(np.log(xtr + EPS), ytr, 1)
    return resolution(np.exp(a * np.log(xte + EPS) + b), Ete)

{"B0_sum": calib(D["sumE"][rtr], D["sumE"][rte], y[rtr], Et[rte]),
 "B1_seed": calib(D["seedE"][rtr], D["seedE"][rte], y[rtr], Et[rte]),
 "B2_total_energy": calib(D["total_energy"][rtr], D["total_energy"][rte], y[rtr], Et[rte])}

{'B0_sum': {'sigma_eff': 0.0649, 'iqr': 0.0675, 'bias': 0.0208},
 'B1_seed': {'sigma_eff': 0.2251, 'iqr': 0.2202, 'bias': 0.0527},
 'B2_total_energy': {'sigma_eff': 0.0579, 'iqr': 0.0593, 'bias': 0.0184}}

## E0b — BDT on 6 aggregate features

In [4]:
gb = HistGradientBoostingRegressor(max_iter=300, random_state=0).fit(D["agg"][rtr], y[rtr])
resolution(np.exp(gb.predict(D["agg"][rte])), Et[rte])

{'sigma_eff': 0.0563, 'iqr': 0.0518, 'bias': 0.0099}

## E1 — transformer (seed coords)

In [5]:
res_e1, _ = train_eval(Transformer(in_dim), D["tok_seed"], y, rtr, rva, rte, Et, EPOCHS, DEVICE, BATCH)
res_e1

{'sigma_eff': 0.0686, 'iqr': 0.069, 'bias': 0.0779}

## E2 — transformer (cluster coords)

In [6]:
train_eval(Transformer(in_dim), D["tok_cluster"], y, rtr, rva, rte, Et, EPOCHS, DEVICE, BATCH)[0]

{'sigma_eff': 0.0911, 'iqr': 0.0899, 'bias': 0.078}

## E11 — Deep Sets (no attention)

In [7]:
train_eval(DeepSets(in_dim), D["tok_seed"], y, rtr, rva, rte, Et, EPOCHS, DEVICE, BATCH)[0]

{'sigma_eff': 0.1136, 'iqr': 0.1124, 'bias': 0.0202}

## E4 — controlled: R3-only vs all-region on the same R3 test set

In [8]:
all_train = np.setdiff1d(np.arange(len(y)), rte)
r3only, _ = train_eval(Transformer(in_dim), D["tok_seed"], y, rtr, rva, rte, Et, EPOCHS, DEVICE, BATCH)
allreg, _ = train_eval(Transformer(in_dim), D["tok_seed"], y, all_train, rva, rte, Et, EPOCHS, DEVICE, BATCH)
{"R3only_on_rte": r3only["sigma_eff"], "allregions_on_rte": allreg["sigma_eff"]}

{'R3only_on_rte': 0.0734, 'allregions_on_rte': 0.0459}

## E12 — feature ablations (leave-one-out vs E1)

In [9]:
def variant_set(var):
    toks_v = [None] * len(y); nc = 7
    for i in ridx:
        tk, nc = variant_tokens(D["raw"][i], var)
        toks_v[i] = tk
    return toks_v, nc

e12 = {}
for name, var in [("drop_frontback", "drop_fb"), ("raw_energy", "raw_e"),
                  ("drop_region_onehot", "drop_onehot"), ("abs_coords", "abs_coords")]:
    tv, nc = variant_set(var)
    vdim = tv[int(ridx[0])].shape[1]
    e12[name] = train_eval(Transformer(vdim), tv, y, rtr, rva, rte, Et, EPOCHS, DEVICE, BATCH, ncont=nc)[0]["sigma_eff"]
e12

{'drop_frontback': 0.102,
 'raw_energy': 0.0517,
 'drop_region_onehot': 0.0674,
 'abs_coords': 0.0783}

## Robustness — multi-seed (the single numbers above wobble)

Re-run E1, E11 and the ablations over a few seeds; report mean and std. A difference smaller than the std is not real.

In [10]:
def multiseed(make, toks, ncont):
    vals = [train_eval(make(), toks, y, rtr, rva, rte, Et, EPOCHS, DEVICE, BATCH, seed=s, ncont=ncont)[0]["sigma_eff"]
            for s in range(SEEDS)]
    return {"mean": round(float(np.mean(vals)), 4), "std": round(float(np.std(vals)), 4)}

robust = {"E1": multiseed(lambda: Transformer(in_dim), D["tok_seed"], 7),
          "E11": multiseed(lambda: DeepSets(in_dim), D["tok_seed"], 7)}
for name, var in [("drop_frontback", "drop_fb"), ("abs_coords", "abs_coords")]:
    tv, nc = variant_set(var)
    vdim = tv[int(ridx[0])].shape[1]
    robust[name] = multiseed(lambda v=vdim: Transformer(v), tv, nc)
robust

{'E1': {'mean': 0.0665, 'std': 0.0092},
 'E11': {'mean': 0.1032, 'std': 0.0086},
 'drop_frontback': {'mean': 0.0568, 'std': 0.0101},
 'abs_coords': {'mean': 0.0736, 'std': 0.01}}

## Full-scale numbers (from the scripts)

In [11]:
import json
full = json.loads((repo / "reports" / "experiment-results_gpu.json").read_text())["results"]
rob = json.loads((repo / "reports" / "robustness.json").read_text())["robust"]
{"full_E1": full["E1_transformer_seed"], "full_BDT": full["E0b_BDT"],
 "robust_E1": rob["E1_transformer_seed"], "robust_E11": rob["E11_deepsets_seed"]}

{'full_E1': {'sigma_eff': 0.0621, 'iqr': 0.0617, 'bias': 0.0561},
 'full_BDT': {'sigma_eff': 0.0563, 'iqr': 0.0518, 'bias': 0.0099},
 'robust_E1': {'mean': 0.0687,
  'std': 0.0117,
  'vals': [0.0847, 0.0769, 0.0509, 0.0623, 0.0689]},
 'robust_E11': {'mean': 0.1032,
  'std': 0.0086,
  'vals': [0.1013, 0.118, 0.0958, 0.1067, 0.0942]}}

## Results summary — all experiments

Two tables. The **single 100-file runs** are one training each (they wobble ~the E1 std). The **5-seed** table is the reliable one — trust it for any claim. It now includes the **tuned transformer** (bigger model + early-stopping + calibration): `TUNED_transformer_allregion` = **0.045 +/- 0.001**, which beats the BDT bar (0.056) at ~11 sigma.

In [1]:
import json
import pandas as pd
from pathlib import Path

repo = Path.cwd().resolve()
if repo.name == "notebooks":
    repo = repo.parent
full = json.loads((repo / "reports" / "experiment-results_gpu.json").read_text())["results"]
pd.DataFrame([{"experiment": k, "sigma_eff": v["sigma_eff"], "iqr": v["iqr"], "bias": v["bias"]}
             for k, v in full.items() if isinstance(v, dict)])

,experiment,sigma_eff,iqr,bias
0,E0_B0_sum_calib,0.0649,0.0675,0.0208
1,E0_B1_seed_calib,0.2251,0.2202,0.0527
2,E0_B2_total_energy_calib,0.0579,0.0593,0.0184
3,E0b_BDT,0.0563,0.0518,0.0099
4,E1_transformer_seed,0.0621,0.0617,0.0561
5,E2_transformer_cluster,0.0911,0.0899,0.0780
6,E11_deepsets_seed,0.1136,0.1124,0.0202
7,E4_all_regions_overall,0.0888,0.0883,0.0777
8,E4_all_regions_on_R3,0.0405,0.0399,0.0425
9,E12a_drop_frontback,0.1020,0.1084,0.0381


### 5-seed robustness (mean +/- std) — the reliable numbers

In [2]:
import json
import pandas as pd
from pathlib import Path

repo = Path.cwd().resolve()
if repo.name == "notebooks":
    repo = repo.parent
rob = json.loads((repo / "reports" / "robustness.json").read_text())
rows = [{"experiment": k, "mean_sigma_eff": v["mean"], "std": v["std"]} for k, v in rob["robust"].items()]
rows += [{"experiment": "E4_" + k, "mean_sigma_eff": v["mean"], "std": v["std"]} for k, v in rob["controlled_E4"].items()]
tuned = json.loads((repo / "reports" / "tuned_3x3_allregion.json").read_text())["tuned_transformer"]
abl = json.loads((repo / "reports" / "tuned_ablation.json").read_text())["ablation"]
rows.append({"experiment": "TUNED_transformer_allregion", "mean_sigma_eff": tuned["mean"], "std": tuned["std"]})
rows.append({"experiment": "TUNED_transformer_r3only", "mean_sigma_eff": abl["B_transformer_r3only"]["mean"], "std": abl["B_transformer_r3only"]["std"]})
rows.append({"experiment": "TUNED_deepsets_allregion", "mean_sigma_eff": abl["C_deepsets_allregion"]["mean"], "std": abl["C_deepsets_allregion"]["std"]})
pd.DataFrame(rows).sort_values("mean_sigma_eff").reset_index(drop=True)

,experiment,mean_sigma_eff,std
0,TUNED_transformer_allregion,0.0453,0.0010
1,E4_allregions_on_rte,0.0504,0.0050
2,TUNED_transformer_r3only,0.0555,0.0021
3,E12a_drop_frontback,0.0568,0.0101
4,E12b_raw_energy,0.0636,0.0124
5,E4_R3only_on_rte,0.0638,0.0087
6,E1_transformer_seed,0.0687,0.0117
7,TUNED_deepsets_allregion,0.0703,0.0032
8,E12c_drop_region_onehot,0.0713,0.0163
9,E12d_abs_coords,0.0736,0.0100


## Resolution comparison (interactive)

5-seed mean +/- std per experiment (lower is better). Green bars beat the BDT bar; the dashed and dotted lines mark the BDT and LHCb `total_energy` references. Saved to `reports/resolution_5seed.html` (interactive) and `.png`.

In [1]:
import json
import plotly.graph_objects as go
from pathlib import Path

repo = Path.cwd().resolve()
if repo.name == "notebooks":
    repo = repo.parent
rob = json.loads((repo / "reports" / "robustness.json").read_text())
full = json.loads((repo / "reports" / "experiment-results_gpu.json").read_text())["results"]

data = list(rob["robust"].items()) + [("E4_" + k, v) for k, v in rob["controlled_E4"].items()]
tuned = json.loads((repo / "reports" / "tuned_3x3_allregion.json").read_text())["tuned_transformer"]
abl = json.loads((repo / "reports" / "tuned_ablation.json").read_text())["ablation"]
data += [("TUNED_transformer_allregion", tuned),
         ("TUNED_transformer_r3only", abl["B_transformer_r3only"]),
         ("TUNED_deepsets_allregion", abl["C_deepsets_allregion"])]
data.sort(key=lambda kv: kv[1]["mean"])
labels = [k for k, _ in data]
means = [v["mean"] for _, v in data]
stds = [v["std"] for _, v in data]
bdt = full["E0b_BDT"]["sigma_eff"]
te = full["E0_B2_total_energy_calib"]["sigma_eff"]
colors = ["#2ca02c" if m < bdt else "#8c8c8c" for m in means]

fig = go.Figure(go.Bar(
    x=means, y=labels, orientation="h",
    error_x=dict(type="data", array=stds, color="#333"),
    marker_color=colors,
    hovertemplate="%{y}<br>sigma_eff = %{x:.4f}<extra></extra>"))
fig.add_vline(x=bdt, line_dash="dash", line_color="crimson",
              annotation_text=f"BDT {bdt:.3f}", annotation_position="top left")
fig.add_vline(x=te, line_dash="dot", line_color="darkorange",
              annotation_text=f"LHCb {te:.3f}", annotation_position="top right")
fig.update_layout(
    title=dict(text="R3 energy resolution: 5-seed mean +/- std (incl. tuned transformer)<br><sub>lower is better; green beats the BDT bar</sub>",
               font=dict(size=15)),
    xaxis_title="sigma_eff   (relative energy resolution)",
    template="plotly_white", width=900, height=500, margin=dict(l=200, r=60, t=90, b=60))
fig.update_yaxes(autorange="reversed")
fig.write_html(str(repo / "reports" / "resolution_5seed.html"))
try:
    fig.write_image(str(repo / "reports" / "resolution_5seed.png"), scale=2)
except Exception:
    pass
fig

/tmp/ipykernel_782439/1369936739.py:42: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(str(repo / "reports" / "resolution_5seed.png"), scale=2)
